## Cell 1 — Install ONNX tools

In [1]:
import subprocess, sys

pkgs = [
    'onnx',
    'onnxruntime',          # CPU inference + quantization tools
    'onnxsim',              # simplify graph — smaller + faster
]
for p in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', p, '-q'])
import onnx
import onnxscript
import onnxruntime as ort
print(f'ONNX version          : {onnx.__version__}')
print(f'ONNXRuntime version   : {ort.__version__}')
print('\n✅ Ready')

ONNX version          : 1.21.0
ONNXRuntime version   : 1.24.4

✅ Ready


In [2]:
import json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import timm
from pathlib import Path

PROJECT_ROOT    = Path(r"D:\rural diagnosis\rural_health_ai")
BEST_MODEL_PATH = PROJECT_ROOT / 'models' / 'checkpoints' / 'best_model.pth'
ONNX_DIR        = PROJECT_ROOT / 'models' / 'onnx'
ONNX_DIR.mkdir(parents=True, exist_ok=True)

FP32_PATH  = ONNX_DIR / 'model_fp32.onnx'
QUANT_PATH = ONNX_DIR / 'model_quantized.onnx'

with open(PROJECT_ROOT / 'data' / 'config.json') as f:
    cfg = json.load(f)

IDX_TO_CLASS = {int(k): v for k, v in cfg['idx_to_class'].items()}

# Load model on CPU for export
# IMPORTANT: always export from CPU, not CUDA
device = torch.device('cpu')
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device,weights_only=False)

model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=7)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print(f'Model loaded: epoch {checkpoint["epoch"]}, '
      f'val acc {checkpoint["val_acc"]*100:.2f}%')
print(f'Device: {device}  (CPU required for ONNX export)')

# Dummy input — same shape the model expects
dummy_input = torch.randn(1, 3, 224, 224)

# Verify PyTorch output before export
with torch.no_grad():
    pt_out = model(dummy_input)
    pt_probs = pt_out.softmax(dim=1)[0]

pt_pred = IDX_TO_CLASS[pt_probs.argmax().item()]
print(f'\nPyTorch test output shape : {pt_out.shape}')
print(f'PyTorch predicted class   : {pt_pred}')
print(f'PyTorch top prob          : {pt_probs.max().item()*100:.2f}%')


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\rural diagnosis\\rural_health_ai\\models\\checkpoints\\best_model.pth'

In [ ]:
print('Exporting to ONNX FP32...')

torch.onnx.export(
    model,
    dummy_input,
    str(FP32_PATH),

    # Operator set version — 17 has best browser support
    opset_version=17,

    # Dynamic batch size — can run 1 image or many
    dynamic_axes={
        'input' : {0: 'batch_size'},
        'output': {0: 'batch_size'}
    },

    input_names  = ['input'],
    output_names = ['output'],

    # Include all ops needed
    do_constant_folding=True,
    export_params=True,
)

fp32_size = FP32_PATH.stat().st_size / 1e6
print(f'✅ FP32 exported: {FP32_PATH.name}')
print(f'   Size: {fp32_size:.1f} MB')

# Verify the exported model is valid
onnx_model = onnx.load(str(FP32_PATH))
onnx.checker.check_model(onnx_model)
print('   Validity check: PASSED')

In [3]:
try:
    from onnxsim import simplify

    print('Simplifying ONNX graph...')
    model_simplified, check = simplify(onnx_model)

    if check:
        onnx.save(model_simplified, str(FP32_PATH))
        new_size = FP32_PATH.stat().st_size / 1e6
        print(f'✅ Graph simplified')
        print(f'   New size: {new_size:.1f} MB (was {fp32_size:.1f} MB)')
    else:
        print('⚠️  Simplification check failed — keeping original')

except Exception as e:
    print(f'onnxsim not available or failed: {e}')
    print('Continuing without simplification — not critical')

Simplifying ONNX graph...
onnxsim not available or failed: name 'onnx_model' is not defined
Continuing without simplification — not critical


In [6]:
from onnxruntime.quantization import quantize_dynamic, QuantType

print('Quantizing to INT8...')
print('(This takes 2-5 minutes)')

quantize_dynamic(
    model_input   = str(FP32_PATH),
    model_output  = str(QUANT_PATH),
    weight_type   = QuantType.QInt8,

    # Quantize all supported ops
    # per_channel=True gives better accuracy on conv layers
    per_channel   = True,
    reduce_range  = True,   # better compatibility on older CPUs
)

quant_size = QUANT_PATH.stat().st_size / 1e6
fp32_final = FP32_PATH.stat().st_size / 1e6

print(f'\n✅ Quantization complete')
print(f'   FP32 model : {fp32_final:.1f} MB')
print(f'   INT8 model : {quant_size:.1f} MB')
print(f'   Compression: {fp32_final/quant_size:.1f}x smaller')

if quant_size < 20:
    print(f'   ✅ Size target met (<20MB)')
else:
    print(f'   ⚠️  Still over 20MB — acceptable, browser can load it')

Quantizing to INT8...
(This takes 2-5 minutes)



✅ Quantization complete
   FP32 model : 16.0 MB
   INT8 model : 4.2 MB
   Compression: 3.8x smaller
   ✅ Size target met (<20MB)


In [5]:
# The dynamic quantization broke conv layers badly
# Use static quantization with calibration data instead
# OR just ship FP32 — it's 21MB and 330ms which is perfectly fine

from onnxruntime.quantization import quantize_dynamic, QuantType
from pathlib import Path

PROJECT_ROOT = Path(r"D:\rural diagnosis\rural_health_ai")
FP32_PATH    = PROJECT_ROOT / 'models' / 'onnx' / 'model_fp32.onnx'
QUANT_PATH   = PROJECT_ROOT / 'models' / 'onnx' / 'model_quantized.onnx'

# Try with per_channel=False — safer for EfficientNet
quantize_dynamic(
    model_input  = str(FP32_PATH),
    model_output = str(QUANT_PATH),
    weight_type  = QuantType.QUInt8,  # UInt8 instead of Int8
    per_channel  = False,             # safer
    reduce_range = False,
)

import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession(str(QUANT_PATH))
test = np.random.randn(1,3,224,224).astype(np.float32)
out  = sess.run(None, {'input': test})[0]
print(f'Output shape: {out.shape}')
print(f'Quantized size: {QUANT_PATH.stat().st_size/1e6:.1f}MB')

Output shape: (1, 7)
Quantized size: 4.2MB


In [6]:
import pandas as pd
import torchvision.transforms as transforms
from PIL import Image
from tqdm import tqdm

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(cfg['imagenet_mean'], cfg['imagenet_std']),
])

# Load 100 test images
test_df = pd.read_csv(PROJECT_ROOT / 'data' / 'test_split.csv')
sample  = test_df.sample(n=100, random_state=42)

# Build sessions
sess_opts = ort.SessionOptions()
sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

fp32_sess  = ort.InferenceSession(str(FP32_PATH),  sess_opts)
quant_sess = ort.InferenceSession(str(QUANT_PATH), sess_opts)

input_name = fp32_sess.get_inputs()[0].name

fp32_correct  = 0
quant_correct = 0
match_count   = 0   # how often both models agree
latencies_fp32  = []
latencies_quant = []

print('Running 100-image accuracy check...')

for _, row in tqdm(sample.iterrows(), total=100):
    true_label = int(row['label'])

    try:
        img = Image.open(row['img_path']).convert('RGB')
    except:
        continue

    tensor = val_tf(img).unsqueeze(0).numpy()   # [1,3,224,224] float32

    # FP32
    t0 = time.time() if False else __import__('time').time()
    fp32_out  = fp32_sess.run(None,  {input_name: tensor})[0]
    latencies_fp32.append((__import__('time').time() - t0) * 1000)

    # Quantized
    t0 = __import__('time').time()
    quant_out = quant_sess.run(None, {input_name: tensor})[0]
    latencies_quant.append((__import__('time').time() - t0) * 1000)

    fp32_pred  = fp32_out[0].argmax()
    quant_pred = quant_out[0].argmax()

    if fp32_pred  == true_label: fp32_correct  += 1
    if quant_pred == true_label: quant_correct += 1
    if fp32_pred  == quant_pred: match_count   += 1

print(f'  FP32  accuracy   : {fp32_correct}%')
print(f'  INT8  accuracy   : {quant_correct}%')
print(f'  Accuracy drop    : {fp32_correct - quant_correct}%')
print(f'  Model agreement  : {match_count}%  (how often both predict same class)')
print(f'  FP32  avg latency: {np.mean(latencies_fp32):.1f}ms per image')
print(f'  INT8  avg latency: {np.mean(latencies_quant):.1f}ms per image')
print(f'  Speed improvement: {np.mean(latencies_fp32)/np.mean(latencies_quant):.1f}x faster')


Running 100-image accuracy check...


100%|██████████| 100/100 [00:12<00:00,  8.23it/s]


─────────────────────────────────────────────
  FP32  accuracy   : 78%
  INT8  accuracy   : 69%
  Accuracy drop    : 9%
  Model agreement  : 59%  (how often both predict same class)
  FP32  avg latency: 13.3ms per image
  INT8  avg latency: 73.4ms per image
  Speed improvement: 0.2x faster
─────────────────────────────────────────────

⚠️  Accuracy drop > 3% — use FP32 model instead for better results
   Person 2 can use model_fp32.onnx in FastAPI if quantized is worse


In [8]:
import time

# Single-threaded = phone CPU simulation
phone_opts = ort.SessionOptions()
phone_opts.intra_op_num_threads = 1
phone_opts.inter_op_num_threads = 1
phone_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

phone_sess = ort.InferenceSession(str(QUANT_PATH), phone_opts)

# Warmup
test_input = np.random.randn(1, 3, 224, 224).astype(np.float32)
for _ in range(3):
    phone_sess.run(None, {input_name: test_input})

# Benchmark 20 runs
times = []
for _ in range(20):
    t0 = time.perf_counter()
    phone_sess.run(None, {input_name: test_input})
    times.append((time.perf_counter() - t0) * 1000)

avg_ms = np.mean(times)
p95_ms = np.percentile(times, 95)

print('Phone CPU simulation (single-threaded):')
print(f'  Average latency : {avg_ms:.1f}ms  ({avg_ms/1000:.2f}s)')
print(f'  P95 latency     : {p95_ms:.1f}ms  ({p95_ms/1000:.2f}s)')
print()



Phone CPU simulation (single-threaded):
  Average latency : 333.8ms  (0.33s)
  P95 latency     : 359.2ms  (0.36s)

✅ Under 2 seconds — good for demo on a phone


In [9]:
# Verify output format matches what Person 2 and Person 3 expect
# import softmax_fn

def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

# Run one real skin image through quantized model
test_df  = pd.read_csv(PROJECT_ROOT / 'data' / 'test_split.csv')
test_row = test_df.iloc[0]

img    = Image.open(test_row['img_path']).convert('RGB')
tensor = val_tf(img).unsqueeze(0).numpy()

raw_out = quant_sess.run(None, {input_name: tensor})[0][0]  # [7] logits
probs   = softmax(raw_out)
top3    = probs.argsort()[::-1][:3]

print('Sample output from quantized model:')
print(f'  True class : {IDX_TO_CLASS[int(test_row["label"])]}')
print()
print('  Top-3 predictions:')
for rank, idx in enumerate(top3):
    cls  = IDX_TO_CLASS[int(idx)]
    conf = probs[idx] * 100
    print(f'  [{rank+1}] {cls:6s}  {conf:5.1f}%')

print()
print('=' * 55)
print('  HANDOFF SUMMARY — what you give your teammates')
print('=' * 55)

fp32_mb  = FP32_PATH.stat().st_size  / 1e6
quant_mb = QUANT_PATH.stat().st_size / 1e6

print(f'\n  FOR PERSON 2 (Backend/FastAPI):')
print(f'    models/onnx/model_fp32.onnx      {fp32_mb:.1f} MB')
print(f'    models/onnx/model_quantized.onnx {quant_mb:.1f} MB')
print(f'    app/backend/gradcam_fn.py        (from Day 3)')
print(f'    data/class_info.json             (Hindi names, next steps)')
print()
print(f'  FOR PERSON 3 (Frontend/PWA):')
print(f'    models/onnx/model_quantized.onnx {quant_mb:.1f} MB')
print(f'    → load with onnxruntime-web in browser')
print(f'    → cache in service worker for offline use')
print(f'    Input  : Float32Array [1, 3, 224, 224]')
print(f'    Output : Float32Array [1, 7] logits')
print(f'    Preprocessing: resize 224×224, ImageNet normalise')
print(f'    Mean: {cfg["imagenet_mean"]}')
print(f'    Std : {cfg["imagenet_std"]}')
print()
print('DAY 4 COMPLETE ✦')
print()
print('Your ML work as Person 1 is now essentially done.')
print('Next: Day 5 — FastAPI backend (Person 2 leads, you support)')

Sample output from quantized model:
  True class : nv

  Top-3 predictions:
  [1] nv      100.0%
  [2] vasc      0.0%
  [3] df        0.0%

  HANDOFF SUMMARY — what you give your teammates

  FOR PERSON 2 (Backend/FastAPI):
    models/onnx/model_fp32.onnx      16.0 MB
    models/onnx/model_quantized.onnx 4.2 MB
    app/backend/gradcam_fn.py        (from Day 3)
    data/class_info.json             (Hindi names, next steps)

  FOR PERSON 3 (Frontend/PWA):
    models/onnx/model_quantized.onnx 4.2 MB
    → load with onnxruntime-web in browser
    → cache in service worker for offline use
    Input  : Float32Array [1, 3, 224, 224]
    Output : Float32Array [1, 7] logits
    Preprocessing: resize 224×224, ImageNet normalise
    Mean: [0.485, 0.456, 0.406]
    Std : [0.229, 0.224, 0.225]

DAY 4 COMPLETE ✦

Your ML work as Person 1 is now essentially done.
Next: Day 5 — FastAPI backend (Person 2 leads, you support)


## Cell 9 — Save inference spec for Person 3
Person 3 needs exact preprocessing steps to match Python output in the browser.

In [10]:
# This JSON tells Person 3 exactly how to preprocess images in JavaScript
inference_spec = {
    'model_file'    : 'model_quantized.onnx',
    'input_name'    : 'input',
    'output_name'   : 'output',
    'input_shape'   : [1, 3, 224, 224],
    'input_dtype'   : 'float32',
    'preprocessing' : {
        'resize'    : [224, 224],
        'channel_order': 'RGB',
        'normalize' : True,
        'mean'      : cfg['imagenet_mean'],
        'std'       : cfg['imagenet_std'],
        'scale'     : '0-1 before normalizing (divide pixels by 255)',
    },
    'output'        : {
        'shape'     : [1, 7],
        'type'      : 'logits (apply softmax to get probabilities)',
        'classes'   : [IDX_TO_CLASS[i] for i in range(7)],
    },
    'class_index_map': IDX_TO_CLASS,
    'confidence_threshold': 0.60,
    'uncertain_message_hindi': 'अनिश्चित — डॉक्टर को दिखाएं',
}

spec_path = PROJECT_ROOT / 'models' / 'onnx' / 'inference_spec.json'
with open(spec_path, 'w', encoding='utf-8') as f:
    json.dump(inference_spec, f, indent=2, ensure_ascii=False)

print(f'✅ Inference spec saved: {spec_path}')
print()
print('Share this file with Person 3 along with model_quantized.onnx')
print('It tells them exactly how to preprocess images in JavaScript')

✅ Inference spec saved: D:\rural diagnosis\rural_health_ai\models\onnx\inference_spec.json

Share this file with Person 3 along with model_quantized.onnx
It tells them exactly how to preprocess images in JavaScript


Output shape: (1, 7)
Quantized size: 4.2MB
